# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone -q https://github.com/rekhasreebandaru/flyrank-ml-internship.git
import pandas as pd
import numpy as np

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

# Rebuild the label and baseline exactly as in ML-07
df["declined"] = (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int)
weak_position = df["position_tier"].isin(["deep", "page_3_5"]).astype(int)
high_volume = (df["search_volume"] >= df["search_volume"].median()).astype(int)
df["baseline_score"] = weak_position * (1 + high_volume) * df["search_volume"]

print(df.shape)
print("Base rate of declined:", df["declined"].mean().round(3))

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
(30000, 46)
Base rate of declined: 0.227


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
print("Method: Logistic Regression first, then Random Forest.")
print("Why: My question is yes/no with an observed label (declined), matching the skill's")
print("first row exactly. Logistic Regression gives a readable starting point; Random Forest")
print("is added only if it earns its complexity over the simpler model.")

Method: Logistic Regression first, then Random Forest.
Why: My question is yes/no with an observed label (declined), matching the skill's
first row exactly. Logistic Regression gives a readable starting point; Random Forest
is added only if it earns its complexity over the simpler model.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from sklearn.model_selection import GroupShuffleSplit

features = ["search_volume", "competition", "cpc", "word_count", "content_age_days",
            "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate"]

df_model = df.dropna(subset=features + ["declined", "client_id"]).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df_model, groups=df_model["client_id"]))
train, test = df_model.iloc[train_idx], df_model.iloc[test_idx]

print("Split design: grouped by client_id, not random rows.")
print("Why: rows from the same client_id are related (same site's content). A random split")
print("would let the model see other pages from the same client during training, then get")
print("evaluated on more of that client's pages in test — leaking client-specific patterns")
print("rather than testing genuine generalization.")
print()
print(f"Train clients: {train['client_id'].nunique()}, Test clients: {test['client_id'].nunique()}")
print(f"Train rows: {len(train)}, Test rows: {len(test)}")

Split design: grouped by client_id, not random rows.
Why: rows from the same client_id are related (same site's content). A random split
would let the model see other pages from the same client during training, then get
evaluated on more of that client's pages in test — leaking client-specific patterns
rather than testing genuine generalization.

Train clients: 21, Test clients: 8
Train rows: 14152, Test rows: 5745


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

X_train, y_train = train[features], train["declined"]
X_test, y_test = test[features], test["declined"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
log_reg_scores = log_reg.predict_proba(X_test_scaled)[:, 1]

rf = RandomForestClassifier(random_state=42, n_estimators=200)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

baseline_scores = test["baseline_score"].values
y_test_arr = y_test.values

base_rate = y_test_arr.mean()

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "ML-07 Baseline (rule)", "Logistic Regression", "Random Forest"],
    "precision@20": [
        base_rate,
        precision_at_k(baseline_scores, y_test_arr, 20),
        precision_at_k(log_reg_scores, y_test_arr, 20),
        precision_at_k(rf_scores, y_test_arr, 20),
    ],
    "precision@50": [
        base_rate,
        precision_at_k(baseline_scores, y_test_arr, 50),
        precision_at_k(log_reg_scores, y_test_arr, 50),
        precision_at_k(rf_scores, y_test_arr, 50),
    ],
})

print(comparison.round(3))

                  method  precision@20  precision@50
0     Base rate (random)         0.257         0.257
1  ML-07 Baseline (rule)         0.200         0.240
2    Logistic Regression         0.250         0.160
3          Random Forest         0.300         0.340


Result: Random Forest is the only method that beats both the baseline and the base rate at every K tested (precision@20: 0.300, precision@50: 0.340). Logistic Regression underperforms even random guessing at precision@50 (0.160 vs. base rate 0.257), showing it doesn't generalize well with a linear decision boundary on these features. The ML-07 rule-based baseline sits below the base rate at precision@20 but above it at precision@50 — a mixed result, reported as-is rather than cherry-picked.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
# Feature importance from Random Forest (the winning model)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Top features Random Forest leans on:")
print(importances.round(3))
print()

# Sanity check: is the top feature suspiciously perfect?
top_feature = importances.index[0]
print(f"Top feature: {top_feature}")
print(f"Correlation with label: {test[top_feature].corr(test['declined']):.3f}")
print()

# Show 3 concrete wrong cases
test_with_preds = test.copy()
test_with_preds["rf_pred"] = rf.predict(X_test)
test_with_preds["rf_prob"] = rf_scores

wrong = test_with_preds[test_with_preds["rf_pred"] != test_with_preds["declined"]]
print(f"Total wrong predictions: {len(wrong)} out of {len(test)}")
print()
print("3 concrete wrong cases:")
print(wrong[["content_id", "declined", "rf_pred", "rf_prob"] + features[:4]].head(3).to_string(index=False))

Top features Random Forest leans on:
ctr                       0.279
word_count                0.145
avg_position              0.137
scroll_rate               0.116
engagement_rate           0.096
content_age_days          0.090
competition               0.042
search_volume             0.039
cpc                       0.028
days_since_last_update    0.027
dtype: float64

Top feature: ctr
Correlation with label: 0.149

Total wrong predictions: 1478 out of 5745

3 concrete wrong cases:
          content_id  declined  rf_pred  rf_prob  search_volume  competition  cpc  word_count
content_304f48230142         1        0    0.365           10.0         0.67 2.05      3221.0
content_9aa793d4d895         1        0    0.275            0.0         0.00 0.00      3515.0
content_d4084a4bc775         1        0    0.450          720.0         1.00 1.05      3080.0


What the model leans on: ctr is the strongest signal (importance 0.279), followed by word_count, avg_position, and scroll_rate. This makes sense — engagement signals like CTR and scroll rate plausibly relate to whether a page's traffic later declines, and its correlation with the label (0.149) is modest, not suspiciously perfect, so this doesn't look like leakage.

Where the model is wrong: 1,478 of 5,745 test rows (25.7%) were misclassified. All three example wrong cases are false negatives — the model predicted "won't decline" but the page actually did. Two of the three cases (content_9aa793d4d895 at zero search volume/competition/CPC, and content_304f48230142 at low volume) have very little traffic signal to work with, making them genuinely hard: a page with almost no visibility gives the model little to distinguish it from any other quiet page, yet it still declined. This suggests the model struggles most on low-traffic content, where feature signal is thin regardless of method.

Overall: Random Forest is the only method that beats both the baseline and the base rate at every K tested — a real, if modest, win. Logistic Regression's failure (worse than random at precision@50) is itself a useful finding: this task isn't linearly separable with these features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.